In [ ]:
### Code to save performance matrix for each model
run_name = "all_data"
save_to_fn = r"C:\Users\louis\PycharmProjects\Master_Thesis\Flamant\experiments\dummy_truss\perf_metrics.txt"

def save_performance_metrics(
    model_name,
    best_params,
    r2_train, mse_train, rmse_train, mape_train,
    r2_val, mse_val, rmse_val, mape_val, filename=save_to_fn,
    run_name=run_name,
):

    with open(filename, "a") as f:   # "a" = append mode

        f.write("========================================\n")
        f.write(f"RUN NAME   : {run_name}\n")
        f.write(f"MODEL NAME : {model_name}\n\n")

        f.write("=== BEST HYPERPARAMETERS ===\n")
        f.write(f"{best_params}\n\n")

        f.write("=== TRAIN SET ===\n")
        f.write(f"R2     : {r2_train:.4f}\n")
        f.write(f"MSE    : {mse_train:.4f} MN²\n")
        f.write(f"RMSE   : {rmse_train:.4f} MN\n")
        f.write(f"MAPE   : {mape_train:.4%}\n\n")

        f.write("=== VALIDATION SET ===\n")
        f.write(f"R2     : {r2_val:.4f}\n")
        f.write(f"MSE    : {mse_val:.4f} MN²\n")
        f.write(f"RMSE   : {rmse_val:.4f} MN\n")
        f.write(f"MAPE   : {mape_val:.4%}\n")

        f.write("\n\n")

In [2]:
import torch

from Flamant.default import *
%cd -q {PROJECT_HOME}

from Flamant.dataset import DummyTrussDataset
from sklearn.experimental import enable_halving_search_cv
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import HalvingRandomSearchCV, cross_validate, KFold

from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_percentage_error

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

seed = 42
torch.random.manual_seed(seed)
np.random.seed(seed)

# Set dataset dirs
train_dataset_dir = r"C:\Users\louis\PycharmProjects\Master_Thesis\Flamant\data\dataset\dummy_truss"
val_dataset_dir = r"C:\Users\louis\PycharmProjects\Master_Thesis\Flamant\data\dataset\dummy_truss"

# Import libraries and open data

In [3]:
def convert_dataset(filepath, noise=0):
    ds = DummyTrussDataset(filepath,
                           f_noise_loads=lambda size: np.random.normal(1, noise / 2, size),
                           f_noise_displacement=lambda size: np.random.normal(1, noise / 2, size),
                           f_noise_strain=lambda size: np.random.normal(1, noise / 2, size)
                           )
    # lambda is used to pass a function creating noise of size size

    X = []
    y = []

    for data in ds:
        x_i, y_i, _, _, _, _ = data
        X.append(x_i)
        y.append(y_i)
    print(x_i)

    return np.array(X), np.array(y)


X_train, y_train = convert_dataset(f"{train_dataset_dir}/val_8192.hdf5")
X_validation, y_validation = convert_dataset(f"{val_dataset_dir}/val_512.hdf5")
#print(X_validation[0])
#rint(X_validation[100])
#X_train = np.hstack((X_train, 1 / X_train[:, :5]))
#X_validation = np.hstack((X_validation, 1 / X_validation[:, :5]))

tensor([ 2.0029e-03, -2.2788e-02,  6.5243e-03,  1.3523e-03, -1.7285e-02,
        -1.5929e+07,  4.0058e-04,  9.0428e-04, -1.2113e-03, -1.5933e-03,
         1.1006e-03])
tensor([ 9.6094e-02, -1.0342e-01,  9.8109e-02,  5.5378e-02, -6.5533e-02,
        -1.5576e+07,  1.9219e-02,  4.0307e-04, -2.2802e-03, -1.0155e-03,
         7.5769e-03])


In [16]:
# Testing its noise funciton

def f_noise_loads(size):
    noise = 0.01
    return np.random.normal(1, noise / 2, size)

print(f_noise_loads(5))
print(f_noise_loads(6))


[0.99967556 0.99882957 0.99394617 0.99360606 0.99832661]
[1.00013729 0.99786412 0.99803238 1.00327    0.99772946 0.99275025]


# Regression

In [4]:
from sklearn.experimental import enable_halving_search_cv
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge
from sklearn.model_selection import HalvingRandomSearchCV
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_percentage_error, make_scorer
from scipy.stats import loguniform, uniform
import numpy as np

# Define pipeline
pipeline = Pipeline([
    ('scaler', MinMaxScaler()),         # rescales each feature to [0, 1]
    ('pca', PCA(n_components=0.95)),    # reduce dimensionality by reorienting space, keeping 95% variance
    ('regression', LinearRegression())  # sets model to linear regression
])

# Define search space
param_distributions = {
    'pca__n_components': uniform(0, 1)
}

# Halving Random Search CV # samples random candidates and evaluate them with cross-validation (cheap broad exploration)
search = HalvingRandomSearchCV(
    estimator=pipeline,
    param_distributions=param_distributions,
    scoring="neg_mean_absolute_percentage_error",   # negative because higher is better
    n_candidates=1000,                              # 1000 candidates of configuration
    factor=3,                                       # roughly keep 1/3 each round
    random_state=42,
    cv=10,                              # 10 fold evaluation
    n_jobs=-1                           # uses all available CPU cores
)

# Fit the search
search.fit(X_train, y_train)
# starts the process:
# 1) randomly draw PCA settings candidate
# 2) for each candidate, run the pipeline (including fitting linear regression)
# 3) predict validation fold and compute score
# 4) eliminate poor candidates
# 5) refit until final best candidate remains
# 6) refit best pipeline on final candidate with full training dataset
# search then contains search.best_params_ and search.best_estimator_,
# representing the best parameters and best fitted model

# Best model
best_pipeline = search.best_estimator_

# Predict
y_train_pred = best_pipeline.predict(X_train)
y_val_pred = best_pipeline.predict(X_validation)
# predicting both to check for good generalization (if only train is good, likely overfitting)


# Metrics function ## for performance evaluation
def compute_metrics(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    mse = mean_squared_error(y_true * 1e-6, y_pred * 1e-6)  # MSE in millions
    rmse = np.sqrt(mse)
    mape = mean_absolute_percentage_error(y_true, y_pred)
    return r2, mse, rmse, mape


best_regression_model = best_pipeline

# Compute metrics
r2_train, mse_train, rmse_train, mape_train = compute_metrics(y_train, y_train_pred)
r2_val, mse_val, rmse_val, mape_val = compute_metrics(y_validation, y_val_pred)

# Print results
print("=== BEST HYPERPARAMETERS ===")
print(search.best_params_)

print("\n=== TRAIN SET ===")
print(f"R2     : {r2_train:.4f}")
print(f"MSE    : {mse_train:.4f} MN^2")
print(f"RMSE   : {rmse_train:.4f} MN^2")
print(f"MAPE   : {mape_train:.4%}")

print("\n=== VALIDATION SET ===")
print(f"R2     : {r2_val:.4f}")
print(f"MSE    : {mse_val:.4f} MN^2")
print(f"RMSE   : {rmse_val:.4f} MN")
print(f"MAPE   : {mape_val:.4%}")

model_name = "Linear regression"
best_params = search.best_params_
save_performance_metrics(
    model_name,
    best_params,
    r2_train, mse_train, rmse_train, mape_train,
    r2_val, mse_val, rmse_val, mape_val, filename=save_to_fn,
    run_name=run_name,
)

=== BEST HYPERPARAMETERS ===
{'pca__n_components': np.float64(0.6775643618422824)}

=== TRAIN SET ===
R2     : 0.0458
MSE    : 29892764.0000 MN^2
RMSE   : 5467.4275 MN^2
MAPE   : 153.2905%

=== VALIDATION SET ===
R2     : 0.0482
MSE    : 29817972.0000 MN^2
RMSE   : 5460.5835 MN
MAPE   : 154.8727%


NameError: name 'save_performance_metrics' is not defined

# Ridge regression

In [ ]:
from sklearn.experimental import enable_halving_search_cv
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge
from sklearn.model_selection import HalvingRandomSearchCV
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_percentage_error, make_scorer
from scipy.stats import loguniform, uniform
import numpy as np

# Define pipeline
pipeline = Pipeline([
    ('scaler', MinMaxScaler()),
    ('pca', PCA(n_components=0.95)),
    ('regression', Ridge())
])

# Define search space
param_distributions = {
    'pca__n_components': uniform(0, 1),
    'regression__alpha': loguniform(1e-6, 1e4)
}

# Halving Random Search CV
search = HalvingRandomSearchCV(
    estimator=pipeline,
    param_distributions=param_distributions,
    scoring="neg_mean_absolute_percentage_error",
    n_candidates=1000,
    factor=3,
    random_state=42,
    cv=10,
    n_jobs=-1
)

# Fit the search
search.fit(X_train, y_train)

# Best model
best_pipeline = search.best_estimator_

# Predict
y_train_pred = best_pipeline.predict(X_train)
y_val_pred = best_pipeline.predict(X_validation)


# Metrics function
def compute_metrics(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    mse = mean_squared_error(y_true * 1e-6, y_pred * 1e-6)  # MSE in millions
    rmse = np.sqrt(mse)
    mape = mean_absolute_percentage_error(y_true, y_pred)
    return r2, mse, rmse, mape


best_ridge_model = best_pipeline

# Compute metrics
r2_train, mse_train, rmse_train, mape_train = compute_metrics(y_train, y_train_pred)
r2_val, mse_val, rmse_val, mape_val = compute_metrics(y_validation, y_val_pred)

# Print results
print("=== BEST HYPERPARAMETERS ===")
print(search.best_params_)

print("\n=== TRAIN SET ===")
print(f"R2     : {r2_train:.4f}")
print(f"MSE    : {mse_train:.4f} MN^2")
print(f"RMSE   : {rmse_train:.4f} MN^2")
print(f"MAPE   : {mape_train:.4%}")

print("\n=== VALIDATION SET ===")
print(f"R2     : {r2_val:.4f}")
print(f"MSE    : {mse_val:.4f} MN^2")
print(f"RMSE   : {rmse_val:.4f} MN")
print(f"MAPE   : {mape_val:.4%}")

model_name = "Ridge regression"
best_params = search.best_params_
save_performance_metrics(
    model_name,
    best_params,
    r2_train, mse_train, rmse_train, mape_train,
    r2_val, mse_val, rmse_val, mape_val, filename=save_to_fn,
    run_name=run_name,
)

# Lasso

In [ ]:
from sklearn.experimental import enable_halving_search_cv  # noqa
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import Lasso
from sklearn.model_selection import HalvingRandomSearchCV
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_percentage_error
from scipy.stats import loguniform
import numpy as np

# Define pipeline with Lasso instead of Ridge
pipeline = Pipeline([
    ('scaler', MinMaxScaler()),
    ('pca', PCA(n_components=0.95)),
    ('regression', Lasso(max_iter=10000))
])

# Define hyperparameter search space
param_distributions = {
    'pca__n_components': uniform(0, 1),
    'regression__alpha': loguniform(1e-6, 1e4)
}

# Halving Random Search CV optimizing for MAPE
search = HalvingRandomSearchCV(
    estimator=pipeline,
    param_distributions=param_distributions,
    scoring="neg_mean_absolute_percentage_error",
    n_candidates=1000,
    factor=3,
    random_state=42,
    cv=10,
    n_jobs=-1
)

# Fit the model
search.fit(X_train, y_train)

# Best model
best_pipeline = search.best_estimator_

# Predict
y_train_pred = best_pipeline.predict(X_train)
y_val_pred = best_pipeline.predict(X_validation)


# Evaluation metrics
def compute_metrics(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    mse = mean_squared_error(y_true * 1e-6, y_pred * 1e-6)  # MSE in MN²
    rmse = np.sqrt(mse)
    mape = mean_absolute_percentage_error(y_true, y_pred)
    return r2, mse, rmse, mape


best_lasso_model = best_pipeline

# Compute train/val metrics
r2_train, mse_train, rmse_train, mape_train = compute_metrics(y_train, y_train_pred)
r2_val, mse_val, rmse_val, mape_val = compute_metrics(y_validation, y_val_pred)

# Display results
print("=== BEST HYPERPARAMETERS ===")
print(search.best_params_)

print("\n=== TRAIN SET ===")
print(f"R2     : {r2_train:.4f}")
print(f"MSE    : {mse_train:.4f} MN²")
print(f"RMSE   : {rmse_train:.4f} MN")
print(f"MAPE   : {mape_train:.4%}")

print("\n=== VALIDATION SET ===")
print(f"R2     : {r2_val:.4f}")
print(f"MSE    : {mse_val:.4f} MN²")
print(f"RMSE   : {rmse_val:.4f} MN")
print(f"MAPE   : {mape_val:.4%}")

model_name = "Lasso regression"
best_params = search.best_params_
save_performance_metrics(
    model_name,
    best_params,
    r2_train, mse_train, rmse_train, mape_train,
    r2_val, mse_val, rmse_val, mape_val, filename=save_to_fn,
    run_name=run_name,
)


Random Forest

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor
from sklearn.experimental import enable_halving_search_cv  # noqa
from sklearn.model_selection import HalvingRandomSearchCV
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_percentage_error
from scipy.stats import randint, uniform
import numpy as np

# Define the pipeline
pipeline = Pipeline([
    ('scaler', MinMaxScaler()),
    ('pca', PCA(n_components=0.95)),
    ('regression', RandomForestRegressor(random_state=42))
])

# Define hyperparameter search space
param_distributions = {
    'pca__n_components': uniform(0.8, 0.2),  # tries values between 0.8 and 1.0
    'regression__n_estimators': randint(50, 300),       # number of trees
    'regression__max_depth': randint(5, 50),            # depth of trees
    'regression__min_samples_split': randint(2, 10),    # lower value, more complex trees
    'regression__min_samples_leaf': randint(1, 5)       # higher values reduce overfitting
}

# Define and run the HalvingRandomSearchCV
search = HalvingRandomSearchCV(
    estimator=pipeline,
    param_distributions=param_distributions,
    scoring="neg_mean_absolute_percentage_error",
    n_candidates=1000,
    factor=3,
    random_state=42,
    cv=10,
    n_jobs=-1,
    verbose=1  # optional, to monitor progress
)

# Fit the model
search.fit(X_train, y_train)

# Get best model
best_pipeline = search.best_estimator_

# Predict
y_train_pred = best_pipeline.predict(X_train)
y_val_pred = best_pipeline.predict(X_validation)


# Evaluation metrics
def compute_metrics(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    mse = mean_squared_error(y_true * 1e-6, y_pred * 1e-6)  # MSE in MN²
    rmse = np.sqrt(mse)
    mape = mean_absolute_percentage_error(y_true, y_pred)
    return r2, mse, rmse, mape


best_forest_model = best_pipeline

# Compute metrics
r2_train, mse_train, rmse_train, mape_train = compute_metrics(y_train, y_train_pred)
r2_val, mse_val, rmse_val, mape_val = compute_metrics(y_validation, y_val_pred)

# Print results
print("=== BEST HYPERPARAMETERS ===")
print(search.best_params_)

print("\n=== TRAIN SET ===")
print(f"R2     : {r2_train:.4f}")
print(f"MSE    : {mse_train:.4f} MN²")
print(f"RMSE   : {rmse_train:.4f} MN")
print(f"MAPE   : {mape_train:.4%}")

print("\n=== VALIDATION SET ===")
print(f"R2     : {r2_val:.4f}")
print(f"MSE    : {mse_val:.4f} MN²")
print(f"RMSE   : {rmse_val:.4f} MN")
print(f"MAPE   : {mape_val:.4%}")

model_name = "Random forest"
best_params = search.best_params_
save_performance_metrics(
    model_name,
    best_params,
    r2_train, mse_train, rmse_train, mape_train,
    r2_val, mse_val, rmse_val, mape_val, filename=save_to_fn,
    run_name=run_name,
)


K-nearest neigbors

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsRegressor
from sklearn.experimental import enable_halving_search_cv  # noqa
from sklearn.model_selection import HalvingRandomSearchCV
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_percentage_error
from scipy.stats import randint, uniform
import numpy as np

# Define the pipeline
pipeline = Pipeline([
    ('scaler', MinMaxScaler()),
    ('pca', PCA(n_components=0.95)),
    ('regression', KNeighborsRegressor())
])

# Define hyperparameter search space
param_distributions = {
    'pca__n_components': uniform(0.8, 0.2),  # n_components in [0.8, 1.0]
    'regression__n_neighbors': randint(1, 50),              # number of neighbors
    'regression__weights': ['uniform', 'distance'],         # distance makes closer neighbors contribute more
    'regression__p': [1, 2]  # 1 = Manhattan, 2 = Euclidean # changes what is considered nearest
}

# Halving Random Search
search = HalvingRandomSearchCV(
    estimator=pipeline,
    param_distributions=param_distributions,
    scoring="neg_mean_absolute_percentage_error",
    n_candidates=1000,
    factor=3,
    random_state=42,
    cv=10,
    n_jobs=-1,
    verbose=1  # Optional: see progress
)

# Fit the model
search.fit(X_train, y_train)

# Best model
best_pipeline = search.best_estimator_

# Predict
y_train_pred = best_pipeline.predict(X_train)
y_val_pred = best_pipeline.predict(X_validation)


# Evaluation metrics
def compute_metrics(y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    mse = mean_squared_error(y_true * 1e-6, y_pred * 1e-6)  # MSE in MN²
    rmse = np.sqrt(mse)
    mape = mean_absolute_percentage_error(y_true, y_pred)
    return r2, mse, rmse, mape


# Optional: save model for later
best_knn_model = best_pipeline

# Compute metrics
r2_train, mse_train, rmse_train, mape_train = compute_metrics(y_train, y_train_pred)
r2_val, mse_val, rmse_val, mape_val = compute_metrics(y_validation, y_val_pred)

# Print results
print("=== BEST HYPERPARAMETERS ===")
print(search.best_params_)

print("\n=== TRAIN SET ===")
print(f"R2     : {r2_train:.4f}")
print(f"MSE    : {mse_train:.4f} MN²")
print(f"RMSE   : {rmse_train:.4f} MN")
print(f"MAPE   : {mape_train:.4%}")

print("\n=== VALIDATION SET ===")
print(f"R2     : {r2_val:.4f}")
print(f"MSE    : {mse_val:.4f} MN²")
print(f"RMSE   : {rmse_val:.4f} MN")
print(f"MAPE   : {mape_val:.4%}")

model_name = "KNN"
best_params = search.best_params_
save_performance_metrics(
    model_name,
    best_params,
    r2_train, mse_train, rmse_train, mape_train,
    r2_val, mse_val, rmse_val, mape_val, filename=save_to_fn,
    run_name=run_name,
)


# Plot

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Define the noise levels
noise_levels = np.linspace(0, 0.1, 11)

# Store metrics for each model
model_names = ['Linear', 'Ridge', 'Lasso', 'Random Forest', 'KNN']
models = [best_regression_model, best_ridge_model, best_lasso_model, best_forest_model, best_knn_model]

r2_scores = {name: [] for name in model_names}
rmse_scores = {name: [] for name in model_names}
mape_scores = {name: [] for name in model_names}
X_val_clean, y_val_clean = convert_dataset(f"{val_dataset_dir}/val_512.hdf5", noise=0)
# Load and evaluate each model on noisy validation data
for noise in noise_levels:
    X_val_noisy, y_val_noisy = convert_dataset(f"{val_dataset_dir}/val_512.hdf5", noise=noise)

    for name, model in zip(model_names, models):
        y_pred = model.predict(X_val_noisy)
        if name == "Linear":
            print(y_pred)

        r2 = r2_score(y_val_noisy, y_pred)
        mse = mean_squared_error(y_val_noisy * 1e-6, y_pred * 1e-6)
        rmse = np.sqrt(mse)
        mape = mean_absolute_percentage_error(y_val_noisy, y_pred)

        r2_scores[name].append(r2)
        rmse_scores[name].append(rmse)
        mape_scores[name].append(mape)

        # to test noise, but not sure it's written well
        print(f"Noise={noise:.2f} | Mean relative change in X: "
        f"{np.mean(np.abs((X_val_noisy - X_val_clean) / X_val_clean)):.4f}")
def save_tested_metrics(run_name, noise_levels,
                        r2_scores, rmse_scores, mape_scores,
                        filename="tested_metrics.txt"):

    with open(filename, "a") as f:
        f.write("="*60 + "\n")
        f.write(f"RUN: {run_name}\n")
        f.write("="*60 + "\n\n")

        for model_name in r2_scores.keys():
            f.write(f"MODEL: {model_name}\n")
            f.write("-"*40 + "\n")

            for i, noise in enumerate(noise_levels):
                f.write(f"Noise level: {noise:.3f}\n")
                f.write(f"  R2   : {r2_scores[model_name][i]:.4f}\n")
                f.write(f"  RMSE : {rmse_scores[model_name][i]:.6f}\n")
                f.write(f"  MAPE : {mape_scores[model_name][i]:.4%}\n\n")

            f.write("\n")

        f.write("\n\n")

save_tested_metrics(run_name, noise_levels, r2_scores, rmse_scores, mape_scores)



In [1]:
# === Plotting ===
plt.style.use("seaborn-v0_8-whitegrid")  # clean academic style

fig, ax = plt.subplots(figsize=(5.8, 5))  # Wider figure for space
colors = ["blue", "red", "green", "purple", "brown", "cyan"]
print(np.array(mape_scores["Linear"]))
# MAPE Plot
for i, name in enumerate(model_names):
    ax.plot(noise_levels * 100, np.array(mape_scores[name]) * 100, label=name, color=colors[i])

ax.legend(frameon=True, facecolor="white", edgecolor="black")
ax.grid(True)
# Analytical benchmark
analy_data = np.loadtxt(r"/Dummy_Truss\analytical_noise_sensitivity.txt", delimiter=",")
x = analy_data[0:11, 0]   # noise levels
y = analy_data[:, 1]   # mape values
analytical_results = y[0:11]

#analytical_results = np.loadtxt(r'experiments/dummy_truss/analytical_mean_mape.dat')
ax.plot(x, analytical_results, c='red', linestyle='--', label="Analytical")
ax.set_xlim(0,10)

# Labels and formatting
ax.set_ylabel("MAPE [%]", fontsize=10)
ax.set_title("Evolution of MAPE as a function of noise", fontsize=12)
ax.set_xlabel("Noise level [%]", fontsize=10)
ax.grid(True)

# Move legend outside the plot

ax.legend(loc='lower center', bbox_to_anchor=(0.5, -0.4), ncols=3, title="Models", fontsize=10)

plt.tight_layout()  # Make room for legend

filename = r"C:\Users\louis\PycharmProjects\Master_Thesis\figures"
# Save
plt.savefig(f"MAPE_noise_{run_name}.pdf", bbox_inches='tight', transparent=False)#f"{filename}\\03-model_comparison_mape.pdf", bbox_inches='tight', transparent=True)








#plt.savefig(f"{filename}\\03-model_comparison_mape.pdf", bbox_inches='tight', transparent=True)
plt.show()
plt.close()

NameError: name 'plt' is not defined

In [25]:
print(X_val_noisy[0])
print(X_val_noisy[1])

[ 3.68461432e-03 -2.36132182e-02  7.96966348e-03  5.31419273e-03
 -1.33249285e-02 -1.58672370e+07]
[ 2.2393091e-02 -5.4041583e-02  2.8145034e-02 -1.3425271e-02
 -4.7382787e-02 -1.6877746e+07]


In [26]:
import pickle
with open("figures/classic_ml_mape_noise.pickle", 'wb') as f:
    pickle.dump(mape_scores, f)

FileNotFoundError: [Errno 2] No such file or directory: 'figures/classic_ml_mape_noise.pickle'

In [ ]:
u = np.array([[0., 0.],
              [0.0047619, -0.02775441],
              [0.00952381, 0.],
              [0.0047619, -0.01823061]]).reshape(-1)
E = 210.e9
A = .0025
EA = E * A
N = np.array([500, 500, -500 * 2 ** .5, -500 * 2 ** .5, 1000]) * 1e3
strain = N / EA
q = np.zeros((4, 2))
q[1, 1] = -1000e3
q = q.reshape(-1)

x_example = np.array([u[2], u[3], u[4], u[6], u[7], q[3], *strain]).reshape((1, -1))
y_example = np.array([EA] * 5).reshape((1, -1))

In [ ]:
i = 5
x = X_validation[[i]]
y = y_validation[[i]]


def decompose_x(x):
    u = np.zeros((4, 3))
    u[1, [0, 1]] = x[0, [0, 1]]
    u[2, [0]] = x[0, 2]
    u[3, [0, 1]] = x[0, [3, 4]]

    q = {1: [0., x[0, 5]]}

    return u, q

In [ ]:
from figures.manim_custom import *
from manim import *
quality = "ql"
config['background_color'] = WHITE


class TrussTarget(MovingCameraScene):
    def __init__(self, **kargs):
        super().__init__(**kargs)
        self.camera.frame.scale(1.1)
        self.camera.frame.move_to(ORIGIN)

    def construct(self):
        nodes = np.array([(0, 0, 0), (5, 0, 0), (10, 0, 0), (5, 5, 0)], dtype=float)
        supports = {0: (True, True), 2: (False, True)}
        connectivity_matrix = np.array([(0, 1), (1, 2), (2, 3), (3, 0), (1, 3)])

        u, loads = decompose_x(x)

        font_size = 30
        g = Truss(nodes=nodes,
                  connectivity_matrix=connectivity_matrix,
                  supports=supports,
                  loads=loads,
                  displacements=u,
                  support_style={'height': .5},
                  node_label_offsets=[.4 * UP, .4 * (UP + RIGHT), .4 * UP, .4 * UP],
                  member_label_offsets=[.3 * UP, .3 * UP, .3 * UR, .3 * UL, .4 * RIGHT],
                  load_labels=[f'{loads[1][1]/1000:.1f}kN'],
                  load_label_offsets=[1 * RIGHT],
                  load_label_style={'prefix': '\\textbf{', 'suffix': '}'},
                  tip_style={'tip_length': .45, 'tip_width': .3},
                  display_node_labels=True,
                  display_load_labels=True,
                  display_member_labels=True,
                  deformed_style={'dash_length': .15})

        g.move_to(ORIGIN)
        g.overlap_deformation(scale=50)
        g.update()

        self.add(g)

        all_group = get_all_vmobjects(self)
        all_group.center()
        path = "figures/img/03-truss_to_predict.svg"
        all_group.to_svg(path, crop=True)

%manim --format png -ql -v WARNING TrussTarget

In [ ]:
from figures.manim_custom import *
from manim import *
quality = "ql"
config['background_color'] = WHITE


class TrussLinearRegression(MovingCameraScene):
    def __init__(self, **kargs):
        super().__init__(**kargs)
        self.camera.frame.scale(1.1)
        self.camera.frame.move_to(ORIGIN)

    def construct(self):
        nodes = np.array([(0, 0, 0), (5, 0, 0), (10, 0, 0), (5, 5, 0)], dtype=float)
        supports = {0: (True, True), 2: (False, True)}
        connectivity_matrix = np.array([(0, 1), (1, 2), (2, 3), (3, 0), (1, 3)])

        u, loads = decompose_x(x)

        font_size = 30
        g = Truss(nodes=nodes,
                  connectivity_matrix=connectivity_matrix,
                  supports=supports,
                  loads=loads,
                  displacements=u,
                  support_style={'height': .5},
                  node_label_offsets=[.4 * UP, .4 * (UP + RIGHT), .4 * UP, .4 * UP],
                  member_label_offsets=[.3 * UP, .3 * UP, .3 * UR, .3 * UL, .4 * RIGHT],
                  load_labels=[f'{loads[1][1]/1000:.1f}kN'],
                  load_label_offsets=[1 * RIGHT],
                  load_label_style={'prefix': '\\textbf{', 'suffix': '}'},
                  tip_style={'tip_length': .45, 'tip_width': .3},
                  display_node_labels=True,
                  display_load_labels=True,
                  display_member_labels=True,
                  deformed_style={'dash_length': .15})

        g.move_to(ORIGIN)
        g.overlap_deformation(scale=50)
        g.update()

        self.add(g)

        all_group = get_all_vmobjects(self)
        all_group.center()
        path = "figures/img/03-truss_to_predict.svg"
        all_group.to_svg(path, crop=True)

%manim --format png -ql -v WARNING TrussTarget

In [ ]:
y_regression = best_regression_model.predict(x)
y_ridge = best_ridge_model.predict(x)
y_lasso = best_lasso_model.predict(x)
y_knn = best_knn_model.predict(x)
y_random_forest = best_forest_model.predict(x)
np.savetxt("figures/x.dat", x, delimiter=' ')
np.savetxt("figures/true_y.dat", y, delimiter=' ')
np.savetxt("figures/pred_regression.dat", y_regression, delimiter=' ')
np.savetxt("figures/pred_ridge.dat", y_ridge, delimiter=' ')
np.savetxt("figures/pred_lasso.dat", y_lasso, delimiter=' ')
np.savetxt("figures/pred_knn.dat", y_knn, delimiter=' ')
np.savetxt("figures/pred_random_forest.dat", y_random_forest, delimiter=' ')